In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A Logic Puzzle

The following exercise is taken from the book 
<a href="https://www.amazon.de/Logeleien-Zweistein-ihren-Antworten-Wegner/dp/B006YF0VUE">"99 Logeleien von Zweistein"</a>.
This book has been published 1968.  It is written by 
<a href="http://de.wikipedia.org/wiki/Thomas_von_Randow">Thomas von Randow</a>.

---
The gentlemen Amann, Bemann, Cemann and Demann are called - not necessarily in the same order - by their first names Erich, Fritz, Gustav and Heiner. They are all married to exactly one woman. We also know the following about them and their wives:

- Either Amann's first name is Heiner, or Bemann's wife is Inge.
- If Cemann is married to Josefa, then - **and only in this case** - Klara's husband is **not** called Fritz.
- If Josefa's husband is **not** called Erich, then Inge is married to Fritz.
- If Luise's husband is called Fritz, then Klara's husband's first name is **not** Gustav.
- If the wife of Fritz is called Inge, then Erich is **not** married to Josefa.
- If Fritz is **not** married to Luise, then Gustav's wife's name is Klara.
- Either Demann is married to Luise, or Cemann is called Gustav.

*What are the full names of these gentlemen, and what are their wives' first names?*

---

We are going to solve this problem by coding it in propositional logic and we will solve the resulting set of clauses using the Davis-Putnam algorithm.  In order to code the problem, we will use the following propositional variables:

- $\texttt{MaleName<}x\texttt{,}z\texttt{>}$ for any male first name $x$ and any surname $z$ expresses
  that the gentleman with first name $x$ has surname $z$.
- $\texttt{Married<}x\texttt{,}y\texttt{>}$ for any male first name $x$ and any female first name $y$ expresses
  that the gentleman with first name $x$ is married to the lady with first name $y$.
- $\texttt{FemaleName<}y\texttt{,}z\texttt{>}$ for any female first name $y$ and any last name $z$ expresses
  that the lady with first name $y$ has surname $z$.

We are using the symbols $\texttt{<}$ and $\texttt{>}$ as part of the propositional variables because we want to show the structure of these variables and the parser for propositional logic accepts these symbols as part of propositional variables.

In [2]:
let FirstMale   = set [ "Erich"; "Fritz"; "Gustav"; "Heiner" ]
let FirstFemale = set [ "Inge"; "Josefa"; "Klara"; "Luise" ]
let SurNames    = set [ "Amann"; "Bemann"; "Cemann"; "Demann" ]

We import the Davis-Putnam algorithm.  This also imports the notebooks `04-CNF.ipynb` and `Propositional-Logic-Parser.ipynb`.

In [3]:
importNotebook "07-Davis-Putnam-JW.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 07-Davis-Putnam-JW.ipynb

The function `makeVar` takes three arguments:
- `f` is the name of a relation, e.g. `"Married"`,
- `x` is a name,
- `y` is a name.

It returns the propositional variable `f<x,y>`.

In [4]:
let makeVar (f: string) (x: string) (y: string) : string =
    f + "<" + x + "," + y + ">"

In [5]:
makeVar "Married" "Heiner" "Klara"

"Married<Heiner,Klara>"

The function `atMostOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of $S$ is `true`.

In [6]:
let atMostOne (S: Set<string>) : CNF =
    set [ for p in S do
            for q in S do
                if p <> q then
                    set [ Neg p; Neg q ] ]

The function `atLeastOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at least one of the variables of $S$ is `true`.

In [7]:
let atLeastOne (S: Set<string>) : CNF =
    set [ Set.map Pos S ]

The function `exactlyOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that exactly one of the variables in the set $S$ is true.

In [8]:
let exactlyOne (S: Set<string>) : CNF =
    atMostOne S + atLeastOne S

In [9]:
exactlyOne (set [ "a"; "b"; "c" ])

set [set [Pos "a"; Pos "b"; Pos "c"]; set [Neg "a"; Neg "b"]; set [Neg "a"; Neg "c"]; set [Neg "b"; Neg "c"]]

The function `bijective` takes three arguments:
- `A` is a set of names,
- `B` is a set of names that has the same number of elements as `A`,
- `f` is the name of a relation.

It returns a set of clauses that is equivalent to the formula
$$   \bigl(\forall x \in A: \exists! y \in B: f\langle x, y\rangle\bigr) \wedge
     \bigl(\forall y \in B: \exists! x \in A: f\langle x, y\rangle\bigr)
$$
Here the expression $f\langle x,y\rangle$ is the name of a propositional variable and the expression $\exists!x:p(x)$ is to be read as "There exists exactly one $x$ such that $p(x)$ holds".

In [10]:
let bijective (A: Set<string>) (B: Set<string>) (f: string) : CNF =
    let mutable Clauses : CNF = Set.empty
    for x in A do
        Clauses <- Clauses + exactlyOne (set [ for y in B -> makeVar f x y ])
    for y in B do
        Clauses <- Clauses + exactlyOne (set [ for x in A -> makeVar f x y ])
    Clauses

In [11]:
printfn "%s" (cnfToString (bijective (set [ "a"; "b"; "c"; "d" ]) (set [ "x"; "y"; "z"; "u" ]) "f"))

{{f<a,u>, f<a,x>, f<a,y>, f<a,z>}, {f<a,u>, f<b,u>, f<c,u>, f<d,u>}, {f<a,x>, f<b,x>, f<c,x>, f<d,x>}, {f<a,y>, f<b,y>, f<c,y>, f<d,y>}, {f<a,z>, f<b,z>, f<c,z>, f<d,z>}, {f<b,u>, f<b,x>, f<b,y>, f<b,z>}, {f<c,u>, f<c,x>, f<c,y>, f<c,z>}, {f<d,u>, f<d,x>, f<d,y>, f<d,z>}, {¬f<a,u>, ¬f<a,x>}, {¬f<a,u>, ¬f<a,y>}, {¬f<a,u>, ¬f<a,z>}, {¬f<a,u>, ¬f<b,u>}, {¬f<a,u>, ¬f<c,u>}, {¬f<a,u>, ¬f<d,u>}, {¬f<a,x>, ¬f<a,y>}, {¬f<a,x>, ¬f<a,z>}, {¬f<a,x>, ¬f<b,x>}, {¬f<a,x>, ¬f<c,x>}, {¬f<a,x>, ¬f<d,x>}, {¬f<a,y>, ¬f<a,z>}, {¬f<a,y>, ¬f<b,y>}, {¬f<a,y>, ¬f<c,y>}, {¬f<a,y>, ¬f<d,y>}, {¬f<a,z>, ¬f<b,z>}, {¬f<a,z>, ¬f<c,z>}, {¬f<a,z>, ¬f<d,z>}, {¬f<b,u>, ¬f<b,x>}, {¬f<b,u>, ¬f<b,y>}, {¬f<b,u>, ¬f<b,z>}, {¬f<b,u>, ¬f<c,u>}, {¬f<b,u>, ¬f<d,u>}, {¬f<b,x>, ¬f<b,y>}, {¬f<b,x>, ¬f<b,z>}, {¬f<b,x>, ¬f<c,x>}, {¬f<b,x>, ¬f<d,x>}, {¬f<b,y>, ¬f<b,z>}, {¬f<b,y>, ¬f<c,y>}, {¬f<b,y>, ¬f<d,y>}, {¬f<b,z>, ¬f<c,z>}, {¬f<b,z>, ¬f<d,z>}, {¬f<c,u>, ¬f<c,x>}, {¬f<c,u>, ¬f<c,y>}, {¬f<c,u>, ¬f<c,z>}, {¬f<c,u>, ¬f<d,u>}, {¬f<c,x

For example, the function call `bijective (set ["a"; "b"; "c"; "d"]) (set ["x"; "y"; "z"; "u"]) "f"` returns a set of 56 clauses: 8 clauses of length 4 and 48 clauses of length 2.

The function `parseAndNormalize` takes one argument:
- `s` is a string representing a propositional formula.

It parses this string as a propositional formula and then turns this formula into a set of clauses.

In [12]:
let parseAndNormalize (s: string) : CNF =
    normalize (parse s)

The function `exclusiveOr` takes two arguments:
- `a` is a string representing a formula,
- `b` is a string representing a formula.

It computes the *exclusive or* of the formulas $a$ and $b$.  The resulting formula is converted into CNF.

In [13]:
let exclusiveOr (a: string) (b: string) : CNF =
    let formula = sprintf "((%s) ↔ ¬(%s))" a b
    parseAndNormalize formula

In [14]:
exclusiveOr "p" "q"

set [set [Pos "p"; Pos "q"]; set [Neg "p"; Neg "q"]]

The function `consistentNames` takes three arguments:
- `FirstMale` is a set of male first names,
- `FirstFemale` is a set of female first names,
- `SurNames` is a set of surnames.

It returns a set of clauses that ensures that if the man `x` has the surname `z` and the woman `y` also has the surname `z`, then `x` and `y` have to be married.

In [15]:
let consistentNames (FirstMale: Set<string>) (FirstFemale: Set<string>) (SurNames: Set<string>) : CNF =
    Set.unionMany [ for x in FirstMale do
                      for y in FirstFemale do
                        for z in SurNames ->
                          parseAndNormalize (sprintf "MaleName<%s,%s> ∧ FemaleName<%s,%s> → Married<%s,%s>" x z y z x y) ]

In [16]:
(consistentNames FirstMale FirstFemale SurNames).Count

64

The function `computeClauses` takes three arguments:
- `FirstMale` is a set of male first names,
- `FirstFemale` is a set of female first names,
- `SurNames` is a set of surnames.

It returns the set of clauses describing the puzzle.

In [17]:
let computeClauses (FirstMale: Set<string>) (FirstFemale: Set<string>) (SurNames: Set<string>) : CNF =
    // Jedem männlichen Vornamen ist genau ein Nachname zugeordnet und umgekehrt.
    let mutable Clauses = bijective FirstMale SurNames "MaleName"
    // Jeder Mann ist mit genau einer Frau verheiratet und umgekehrt.
    Clauses <- Clauses + bijective FirstMale FirstFemale "Married"
    // Jede Frau hat genau einen Nachnamen
    Clauses <- Clauses + bijective FirstFemale SurNames "FemaleName"
    // Die Namen sind konsistent.
    Clauses <- Clauses + consistentNames FirstMale FirstFemale SurNames
    // Entweder ist Amanns Vorname Heiner, oder Bemanns Frau heisst Inge.
    Clauses <- Clauses + exclusiveOr "MaleName<Heiner,Amann>" "FemaleName<Inge,Bemann>"
    // Wenn Cemann mit Josefa verheiratet ist, dann – und nur in diesem Falle –
    // heisst Klaras Mann nicht Fritz.
    Clauses <- Clauses + parseAndNormalize "FemaleName<Josefa,Cemann> ↔ ¬Married<Fritz,Klara>"
    // Wenn Josefas Mann nicht Erich heisst, dann ist Inge mit Fritz verheiratet.
    Clauses <- Clauses + parseAndNormalize "¬Married<Erich,Josefa> → Married<Fritz,Inge>"
    // Wenn Luises Mann Fritz heisst, dann ist der Vorname von Klaras Mann nicht Gustav.
    Clauses <- Clauses + parseAndNormalize "Married<Fritz,Luise> → ¬Married<Gustav,Klara>"
    // Wenn die Frau von Fritz Inge heisst, dann ist Erich nicht mit Josefa verheiratet.
    Clauses <- Clauses + parseAndNormalize "Married<Fritz,Inge> → ¬Married<Erich,Josefa>"
    // Wenn Fritz nicht mit Luise verheiratet ist, dann heisst Gustavs Frau Klara.
    Clauses <- Clauses + parseAndNormalize "¬Married<Fritz,Luise> → Married<Gustav,Klara>"
    // Entweder ist Demann mit Luise verheiratet, oder Cemann heisst Gustav.
    Clauses <- Clauses + exclusiveOr "FemaleName<Luise,Demann>" "MaleName<Gustav,Cemann>"
    Clauses

In [18]:
let Clauses = computeClauses FirstMale FirstFemale SurNames
printfn "%s" (cnfToString Clauses)

{{FemaleName<Inge,Amann>, FemaleName<Inge,Bemann>, FemaleName<Inge,Cemann>, FemaleName<Inge,Demann>}, {FemaleName<Inge,Amann>, FemaleName<Josefa,Amann>, FemaleName<Klara,Amann>, FemaleName<Luise,Amann>}, {FemaleName<Inge,Bemann>, FemaleName<Josefa,Bemann>, FemaleName<Klara,Bemann>, FemaleName<Luise,Bemann>}, {FemaleName<Inge,Bemann>, MaleName<Heiner,Amann>}, {FemaleName<Inge,Cemann>, FemaleName<Josefa,Cemann>, FemaleName<Klara,Cemann>, FemaleName<Luise,Cemann>}, {FemaleName<Inge,Demann>, FemaleName<Josefa,Demann>, FemaleName<Klara,Demann>, FemaleName<Luise,Demann>}, {FemaleName<Josefa,Amann>, FemaleName<Josefa,Bemann>, FemaleName<Josefa,Cemann>, FemaleName<Josefa,Demann>}, {FemaleName<Josefa,Cemann>, Married<Fritz,Klara>}, {FemaleName<Klara,Amann>, FemaleName<Klara,Bemann>, FemaleName<Klara,Cemann>, FemaleName<Klara,Demann>}, {FemaleName<Luise,Amann>, FemaleName<Luise,Bemann>, FemaleName<Luise,Cemann>, FemaleName<Luise,Demann>}, {FemaleName<Luise,Demann>, MaleName<Gustav,Cemann>}, {Mal

There are 242 different clauses.

In [19]:
Clauses.Count

242

The function `computeSolution` takes the same three arguments as the function `computeClauses`.  It computes the clauses describing the puzzle and solves them.  The result is a set of unit clauses.

In [20]:
let computeSolution (FirstMale: Set<string>) (FirstFemale: Set<string>) (SurNames: Set<string>) : CNF =
    let Clauses = computeClauses FirstMale FirstFemale SurNames
    solve Clauses

In [21]:
#!time
let Solution = computeSolution FirstMale FirstFemale SurNames

Wall time: 23.3521ms

The function `onlyPositive` takes one argument:
- `Solution` is a set of unit clauses.

It returns the set of all variables that are true according to `Solution`.  The function `arb` is defined in the notebook `07-Davis-Putnam-JW.ipynb`.

In [22]:
let onlyPositive (Solution: CNF) : Set<string> =
    set [ for Clause in Solution do
            match arb Clause with
            | Pos v -> v
            | Neg _ -> () ]

In [23]:
onlyPositive Solution

set ["FemaleName<Inge,Bemann>"; "FemaleName<Josefa,Cemann>"; "FemaleName<Klara,Amann>"; "FemaleName<Luise,Demann>"; "MaleName<Erich,Demann>"; "MaleName<Fritz,Bemann>"; "MaleName<Gustav,Amann>"; "MaleName<Heiner,Cemann>"; "Married<Erich,Luise>"; "Married<Fritz,Inge>"; "Married<Gustav,Klara>"; "Married<Heiner,Josefa>"]

The functions `extractFirst` and `extractSecond` take one argument:
- `s` is a variable of the form `f<x,y>`.

The function `extractFirst` returns `x`, while `extractSecond` returns `y`.  Both functions use regular expressions.  `Regex.Match` returns the first match of the regular expression in `s`, and `m.Groups[1].Value` is the substring matched by the first parenthesized group of the regular expression.

In [24]:
open System.Text.RegularExpressions

let extractFirst (s: string) : string =
    let m = Regex.Match(s, "<([A-Za-z]+),")
    m.Groups[1].Value

let extractSecond (s: string) : string =
    let m = Regex.Match(s, ",([A-Za-z]+)>")
    m.Groups[1].Value

The function `displaySolution` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the puzzle.

It prints the solution.  The map `Married` maps every male first name to the first name of his wife, the map `Names` maps every male first name to the corresponding surname.

In [25]:
let displaySolution (Solution: CNF) : unit =
    let mutable Married : Map<string, string> = Map.empty
    let mutable Names   : Map<string, string> = Map.empty
    for v in onlyPositive Solution do
        if v.StartsWith "Married" then
            Married <- Married.Add(extractFirst v, extractSecond v)
        elif v.StartsWith "MaleName" then
            Names <- Names.Add(extractFirst v, extractSecond v)
    for KeyValue(x, y) in Married do
        printfn "%s %s is married to %s." x Names[x] y

In [26]:
displaySolution Solution

Erich

Demann

 is married to 

Luise

.

...

Heiner

Cemann

 is married to 

Josefa

.

## Checking the Uniqueness of the Solution

The function `negateSolution` takes one argument:
- `UnitClauses` is a set of unit clauses.

It returns a clause that is the negation of the conjunction of these unit clauses.

In [27]:
let negateSolution (UnitClauses: CNF) : Clause =
    Set.map (fun Unit -> complement (arb Unit)) UnitClauses

In [28]:
negateSolution (set [ set [ Pos "a" ]; set [ Neg "b" ] ])

set [Pos "b"; Neg "a"]

The function `checkUniqueness` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of `Clauses`,
- `Clauses` is a set of clauses.

It checks whether `Solution` is the only solution of `Clauses`.

In [29]:
let checkUniqueness (Solution: CNF) (Clauses: CNF) : unit =
    let negation    = negateSolution Solution
    let alternative = solve (Clauses.Add negation)
    if alternative = set [ Set.empty ] then
        printfn "Well done: The solution is unique!"
    else
        printfn "ERROR: The solution is not unique!"

In [30]:
#!time
checkUniqueness Solution Clauses

Well done: The solution is unique!

Wall time: 69.9612ms